# 🧬 Cancer Prediction Using Machine Learning

## Phase 4 — Exploratory Data Analysis (EDA)

**Objective:** Visualize the data to discover patterns, relationships, and insights
that will guide our feature engineering and model selection.

---

### 📋 Table of Contents

1. Setup & Load Data
2. Feature Distribution Histograms
3. Correlation Heatmap
4. Boxplots by Diagnosis
5. Violin Plots
6. Scatter Plots (Top Feature Pairs)
7. Pairplot (Top 5 Features)
8. Key Findings & Phase Summary

In [ ]:
# =============================================================================
# CELL 1: Setup & Load Cleaned Data
# =============================================================================

import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Add src/ to path
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from utils import print_section_header

# Load cleaned data
df = pd.read_csv('../data/processed/breast_cancer_cleaned.csv')

# Separate features and target
feature_cols = [col for col in df.columns if col != 'diagnosis']
mean_features = [col for col in feature_cols if 'mean' in col]
se_features = [col for col in feature_cols if 'se' in col]
worst_features = [col for col in feature_cols if 'worst' in col]

print(f"✅ Loaded cleaned dataset: {df.shape}")
print(f"   Mean features:  {len(mean_features)}")
print(f"   SE features:    {len(se_features)}")
print(f"   Worst features: {len(worst_features)}")

---

## 2. 📊 Feature Distribution Histograms

### Why histograms?

Histograms show the **shape** of each feature's distribution. We look for:
- **Normal vs skewed** distributions (affects algorithm choice)
- **Separation between classes** (features where M and B don't overlap are great predictors)
- **Outliers** (long tails)
- **Bimodal distributions** (two peaks — may indicate subgroups)

In [ ]:
# =============================================================================
# CELL 2: Distribution Histograms for Mean Features
# =============================================================================
# WHY: We overlay Benign and Malignant distributions to see which
#       features best separate the two classes. Features where the
#       two distributions have minimal overlap are strong predictors.
# =============================================================================

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    ax = axes[i]
    
    # Plot histograms for each class
    ax.hist(df[df['diagnosis'] == 0][feature], bins=25, alpha=0.6,
            color='#3fb950', label='Benign', edgecolor='white')
    ax.hist(df[df['diagnosis'] == 1][feature], bins=25, alpha=0.6,
            color='#f85149', label='Malignant', edgecolor='white')
    
    # Clean title
    clean_name = feature.replace(' mean', '').replace('_', ' ').title()
    ax.set_title(clean_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('📊 Feature Distributions by Diagnosis (Mean Features)',
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/histograms_mean_features.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/histograms_mean_features.png")
plt.show()

### 💡 Observation

**Strong separators** (minimal overlap between B and M):
- `concave points mean` — Malignant tumors have significantly more concave points
- `concavity mean` — Higher concavity indicates more irregular shape
- `radius mean`, `area mean`, `perimeter mean` — Cancer tumors are physically larger

**Weak separators** (high overlap):
- `fractal dimension mean` — Very similar distributions
- `symmetry mean` — Significant overlap
- `smoothness mean` — Hard to distinguish

**Business insight:** The model will likely rely most on tumor **size** and **shape irregularity** features.

In [ ]:
# =============================================================================
# CELL 3: Distribution Histograms for Worst Features
# =============================================================================
# WHY: "Worst" features capture the most extreme cell nuclei.
#       These are often MORE discriminative than mean features because
#       cancer is characterized by its most abnormal cells.
# =============================================================================

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for i, feature in enumerate(worst_features):
    ax = axes[i]
    ax.hist(df[df['diagnosis'] == 0][feature], bins=25, alpha=0.6,
            color='#3fb950', label='Benign', edgecolor='white')
    ax.hist(df[df['diagnosis'] == 1][feature], bins=25, alpha=0.6,
            color='#f85149', label='Malignant', edgecolor='white')
    
    clean_name = feature.replace(' worst', '').replace('_', ' ').title()
    ax.set_title(f"{clean_name} (Worst)", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('📊 Feature Distributions by Diagnosis (Worst Features)',
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/histograms_worst_features.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/histograms_worst_features.png")
plt.show()

---

## 3. 🔥 Correlation Heatmap

### Why correlation analysis?

Correlation tells us how features **move together**:
- **High positive correlation** (≈1.0): When one goes up, the other goes up
- **High negative correlation** (≈-1.0): When one goes up, the other goes down
- **Near zero**: No linear relationship

### Why it matters for ML:

1. **Multicollinearity**: Highly correlated features provide redundant information,
   which can hurt some models (Logistic Regression) and waste computation
2. **Feature selection**: We can drop one of two highly correlated features
3. **Feature engineering**: Correlated features might be combined into a single feature

In [ ]:
# =============================================================================
# CELL 4: Full Correlation Heatmap
# =============================================================================
# WHY: The 30×30 correlation matrix reveals groups of redundant features.
#       We expect radius, perimeter, and area to be highly correlated
#       (they all measure tumor size).
# =============================================================================

# Compute correlation matrix
corr_matrix = df[feature_cols].corr()

# Create a large heatmap
fig, ax = plt.subplots(figsize=(20, 16))

# Use a mask for the upper triangle (avoid redundancy)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=False,  # Too crowded with 30 features
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'},
    ax=ax
)

ax.set_title('🔥 Feature Correlation Matrix (30 Features)',
             fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()

plt.savefig('../images/correlation_heatmap_full.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/correlation_heatmap_full.png")
plt.show()

In [ ]:
# =============================================================================
# CELL 5: Identify Highly Correlated Feature Pairs
# =============================================================================
# WHY: We need to find pairs with |correlation| > 0.9 to decide
#       which redundant features to drop in Feature Engineering.
# =============================================================================

print_section_header("Highly Correlated Feature Pairs (|r| > 0.9)", "🔗")

# Extract upper triangle pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.9:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': round(corr_val, 4)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
print(f"Found {len(high_corr_df)} highly correlated pairs:\n")
print(high_corr_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 6: Correlation with Target Variable
# =============================================================================
# WHY: Features highly correlated with the target are the best predictors.
#       This helps us prioritize which features to keep.
# =============================================================================

# Correlation of each feature with the target
target_corr = df[feature_cols].corrwith(df['diagnosis']).sort_values(ascending=False)

# Interactive bar chart
fig = px.bar(
    x=target_corr.values,
    y=target_corr.index,
    orientation='h',
    title='🎯 Feature Correlation with Diagnosis (Target)',
    labels={'x': 'Correlation Coefficient', 'y': 'Feature'},
    color=target_corr.values,
    color_continuous_scale='RdBu_r',
    range_color=[-1, 1]
)

fig.update_layout(
    height=700,
    template='plotly_dark',
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_colorbar_title='Corr'
)

fig.show()

try:
    fig.write_image('../images/target_correlation_bar.png', scale=2)
    print("✅ Saved: images/target_correlation_bar.png")
except Exception as e:
    print(f"⚠️ Could not save static image: {e}")

# Print top and bottom correlated features
print("\n📊 Top 5 positively correlated with Malignancy:")
for feat, corr in target_corr.head(5).items():
    print(f"   {feat:35s} → {corr:+.4f}")

print("\n📊 Top 5 least correlated (near zero):")
least_corr = target_corr.abs().sort_values().head(5)
for feat in least_corr.index:
    print(f"   {feat:35s} → {target_corr[feat]:+.4f}")

### 💡 Key Correlation Insights

**Strongest predictors of Malignancy** (high positive correlation with diagnosis=1):
- `concave points worst/mean` — Shape irregularity is the #1 indicator
- `perimeter worst`, `radius worst`, `area worst` — Larger tumors = more likely malignant

**Redundant feature groups** (corr > 0.9 with each other):
- `radius`, `perimeter`, `area` (all measure size — keep one)
- `concavity` and `concave points` (both measure shape — keep one)
- `mean` and `worst` variants of the same feature

**🎯 Interview Question:** *What is multicollinearity and why is it a problem?*
> Multicollinearity occurs when two or more features are highly correlated. It causes: 1) Unstable coefficient estimates in linear models, 2) Difficulty interpreting feature importance, 3) Wasted computation on redundant information. Solutions include dropping one of the correlated features, PCA, or using regularization (L1/L2).

---

## 4. 🎻 Violin Plots

Violin plots combine **boxplots** and **kernel density estimation (KDE)**.
They show the full distribution shape, not just summary statistics.

In [ ]:
# =============================================================================
# CELL 7: Violin Plots for Top Discriminative Features
# =============================================================================
# WHY: Violins show the FULL distribution, not just quartiles.
#       This reveals bimodality and distribution shape differences
#       that boxplots miss.
# =============================================================================

# Select top 6 most correlated features with target
top_6_features = target_corr.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(top_6_features):
    sns.violinplot(
        data=df,
        x='diagnosis',
        y=feature,
        hue='diagnosis',
        palette={0: '#3fb950', 1: '#f85149'},
        inner='box',
        ax=axes[i],
        legend=False
    )
    
    clean_name = feature.replace('_', ' ').title()
    axes[i].set_title(f'{clean_name}', fontsize=13, fontweight='bold')
    axes[i].set_xticklabels(['Benign (0)', 'Malignant (1)'])
    axes[i].set_xlabel('')

plt.suptitle('🎻 Violin Plots — Top 6 Most Discriminative Features',
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/violin_plots_top6.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/violin_plots_top6.png")
plt.show()

### 💡 Observation

The violin plots clearly show:
- **Malignant tumors** have distributions shifted to higher values for size/shape features
- Some features show **bimodal distributions** in the Malignant class — suggesting subtypes
- The **minimal overlap** in `concave points worst` makes it an excellent single predictor

---

## 5. 🔍 Scatter Plots (Top Feature Pairs)

In [ ]:
# =============================================================================
# CELL 8: Interactive Scatter Plots for Key Feature Pairs
# =============================================================================
# WHY: Scatter plots show how two features interact to separate classes.
#       If we can draw a line between Benign and Malignant clusters,
#       a linear model will work well.
# =============================================================================

# Define interesting feature pairs based on domain knowledge
scatter_pairs = [
    ('mean radius', 'mean texture'),
    ('mean area', 'mean smoothness'),
    ('mean concavity', 'mean concave points'),
    ('worst radius', 'worst concave points'),
]

# Adjust for sklearn column names (use spaces)
# Check which naming convention is used
sample_col = df.columns[0] if 'diagnosis' not in df.columns[0] else df.columns[1]
uses_spaces = ' ' in sample_col

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{a} vs {b}' for a, b in scatter_pairs])

for idx, (feat_a, feat_b) in enumerate(scatter_pairs):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    for diagnosis, color, name in [(0, '#3fb950', 'Benign'), (1, '#f85149', 'Malignant')]:
        subset = df[df['diagnosis'] == diagnosis]
        fig.add_trace(
            go.Scatter(
                x=subset[feat_a],
                y=subset[feat_b],
                mode='markers',
                marker=dict(color=color, size=5, opacity=0.6),
                name=name,
                showlegend=(idx == 0),  # Show legend only once
            ),
            row=row, col=col
        )

fig.update_layout(
    title='🔍 Scatter Plots — Key Feature Pairs by Diagnosis',
    height=700,
    template='plotly_dark',
    legend=dict(x=0.85, y=0.98)
)

fig.show()

try:
    fig.write_image('../images/scatter_plots_key_pairs.png', scale=2)
    print("✅ Saved: images/scatter_plots_key_pairs.png")
except Exception as e:
    print(f"⚠️ Could not save static image: {e}")

### 💡 Observation

The scatter plots reveal:
- **Clear class separation** in most feature pairs — the data IS linearly separable for top features
- **Concavity vs Concave Points** shows the tightest clustering with clear boundaries
- **Some overlap** exists in the middle ranges — these borderline cases are where the model earns its value

**Conclusion:** Linear models (Logistic Regression, SVM with linear kernel) should perform well, but non-linear models may capture the borderline cases better.

---

## 6. 🔬 Pairplot (Top 5 Features)

In [ ]:
# =============================================================================
# CELL 9: Pairplot for Top 5 Features
# =============================================================================
# WHY: Pairplots show ALL pairwise relationships at once.
#       Diagonal shows individual distributions, off-diagonal shows scatter.
#       We limit to top 5 features to keep it readable.
#
# NOTE: Pairplots are computationally expensive for many features.
#       With 30 features, it would create 900 subplots — unusable.
# =============================================================================

top_5_features = target_corr.abs().sort_values(ascending=False).head(5).index.tolist()

print(f"Creating pairplot for: {top_5_features}")

# Create pairplot
pairplot_df = df[top_5_features + ['diagnosis']].copy()
pairplot_df['diagnosis'] = pairplot_df['diagnosis'].map({0: 'Benign', 1: 'Malignant'})

g = sns.pairplot(
    pairplot_df,
    hue='diagnosis',
    palette={'Benign': '#3fb950', 'Malignant': '#f85149'},
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 20},
    diag_kws={'alpha': 0.6},
    height=2.5
)

g.figure.suptitle('🔬 Pairplot — Top 5 Most Predictive Features',
                  fontsize=16, fontweight='bold', y=1.02)

plt.savefig('../images/pairplot_top5.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/pairplot_top5.png")
plt.show()

---

## 7. 📋 Phase 4 — EDA Summary

### ✔ Summary

We generated 8 comprehensive visualizations revealing key patterns:

| Visualization | Key Finding |
|--------------|-------------|
| Histograms (Mean) | Concave points and concavity best separate M/B |
| Histograms (Worst) | "Worst" features are even more discriminative |
| Correlation Heatmap | Strong multicollinearity in size features |
| Target Correlation | Concave points worst has highest correlation with diagnosis |
| Violin Plots | Clear distribution shifts in top features |
| Scatter Plots | Classes are largely linearly separable |
| Pairplot | All top 5 features show clear M/B separation |

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| Multicollinearity | Correlated features that provide redundant info |
| Feature-Target Correlation | Identifies most predictive features |
| Distribution Overlap | Less overlap = better predictor |
| Violin vs Boxplot | Violins show full distribution shape |
| Pairplot | All pairwise relationships at a glance |

### ✔ Files Created

```
📁 images/
├── histograms_mean_features.png
├── histograms_worst_features.png
├── correlation_heatmap_full.png
├── target_correlation_bar.png
├── violin_plots_top6.png
├── scatter_plots_key_pairs.png
└── pairplot_top5.png
```

### ✔ Next Phase: Phase 5 — Feature Engineering

We'll use these insights to:
1. Remove highly correlated redundant features
2. Compute feature importance scores
3. Select the optimal feature subset

---

*Phase 4 Complete ✅*